<a href="https://colab.research.google.com/github/whatman42/idx/blob/main/colab/IDX_GPU_TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IDX COLAB GPU TRAINING CENTER

**Peran:** training + research eksternal (bukan production runtime)

Architecture freeze: production pointer **tidak** diubah dari Colab secara default.


## 1. Environment Check


In [1]:
import sys, os, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU count:', os.cpu_count())


Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
CPU count: 2


## 2. Connect Google Drive (optional)


In [2]:
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/IDX')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE.mkdir(parents=True, exist_ok=True)
    print('DRIVE = PASS', DRIVE)
except Exception as e:
    print('DRIVE = BLOCKED', type(e).__name__)


Mounted at /content/drive
DRIVE = PASS /content/drive/MyDrive/IDX


## 3. Clone / Update Repository


In [3]:
import os, subprocess
from pathlib import Path
REPO = 'https://github.com/whatman42/idx.git'
ROOT = Path('/content/idx')
if not (ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'pull', '--ff-only'])
os.chdir(ROOT)
print('cwd', os.getcwd())
sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', sha)


cwd /content/idx
Commit: 6ecc91fc3e4b0f7e07a58d15bf30c525f339c8d5


## 4. Install Dependencies


In [4]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('deps installed')


deps installed


## 5. GPU Check


In [5]:
import sys
sys.path.insert(0, '/content/idx')
from colab.environment_check import check_environment
env = check_environment()
print('GPU_AVAILABLE =', env.gpu_available)
print('GPU:', env.gpu_name, 'VRAM GB:', env.gpu_vram_gb)
print('Status:', env.status, env.notes)


GPU_AVAILABLE = False
GPU:  VRAM GB: 0.0
Status: PASS ['GPU_AVAILABLE=false; CPU fallback active']


## 6-16. Training Pipeline


In [6]:
from colab.colab_config import ColabConfig, DatasetType
from colab.colab_train import run_colab_training
cfg = ColabConfig(out_dir='artifacts/colab_candidates', dataset_type=DatasetType.SYNTHETIC_DATA, n_bars=120, promote=False)
report = run_colab_training(cfg)
print(report.summary_text())
print('MARKET PERFORMANCE =', report.market_performance)
print('GPU LIVE =', report.gpu_live)
print('artifact:', report.artifact_dir)


/content/idx/colab/colab_train.py:146: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  data[c] = s.dt.to_pydatetime().tolist() if pd.api.types.is_datetime64_any_dtype(s) else s.to_numpy().tolist()


IDX COLAB GPU TRAINING CENTER
Repository : whatman42/idx
Commit     : 6ecc91fc3e4b0f7e07a58d15bf30c525f339c8d5
Dataset    : SYNTHETIC_DATA
Mode       : TRAINING
GPU LIVE   : BLOCKED
GEMINI LIVE: BLOCKED
TURSO LIVE : BLOCKED
REAL IDX   : BLOCKED
MARKET PERF: UNVERIFIED
Approved   : False
Prod unchanged: True

Status:
[PASS] Environment — GPU_AVAILABLE=false; CPU fallback active
[PASS] Secrets — gemini=absent; turso=absent
[PASS] Dataset — SYNTHETIC_DATA
[PASS] DataLoad — synthetic_rows=120
[PASS] DataQuality — cols=['timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume']
[FAIL] Pipeline — InvalidOperationError: window expression not allowed in aggregation
MARKET PERFORMANCE = UNVERIFIED
GPU LIVE = BLOCKED
artifact: 


## 17. SHA256 verify


In [7]:
from pathlib import Path
from colab.artifact_export import verify_bundle
if report.artifact_dir:
    ok, reason = verify_bundle(Path(report.artifact_dir))
    print('SHA256:', 'PASS' if ok else 'FAIL', reason)
else:
    print('SHA256: SKIP')


SHA256: SKIP


## 18. Final Status


In [8]:
print(report.summary_text())
assert report.production_unchanged
print('PRODUCTION POINTER: UNCHANGED')


IDX COLAB GPU TRAINING CENTER
Repository : whatman42/idx
Commit     : 6ecc91fc3e4b0f7e07a58d15bf30c525f339c8d5
Dataset    : SYNTHETIC_DATA
Mode       : TRAINING
GPU LIVE   : BLOCKED
GEMINI LIVE: BLOCKED
TURSO LIVE : BLOCKED
REAL IDX   : BLOCKED
MARKET PERF: UNVERIFIED
Approved   : False
Prod unchanged: True

Status:
[PASS] Environment — GPU_AVAILABLE=false; CPU fallback active
[PASS] Secrets — gemini=absent; turso=absent
[PASS] Dataset — SYNTHETIC_DATA
[PASS] DataLoad — synthetic_rows=120
[PASS] DataQuality — cols=['timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume']
[FAIL] Pipeline — InvalidOperationError: window expression not allowed in aggregation
PRODUCTION POINTER: UNCHANGED
